In [ ]:
import pandas as pd

# Path to data frame containing all WSIs with a matched rekvnr as well as SNOMED categories
df_path = "D:\DATA\with_snomed_category_new.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

In [ ]:
from helper_functions import strings2lists

list_str_col = ["snomed_code", "M", "T", "snomed_text", "T_text", "M_text", "undersoeger_anonymous", "T_category", "M_category"]
for col in list_str_col: 
    df_all[col]=df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")

In [ ]:
# Check unique counts and normalized categorical distributions to identify class imbalance
for col in df_HE.columns:
    print(df_HE[col].value_counts())
    print("")
    print(df_HE[col].value_counts(normalize=True))
    print("-"*40)

In [ ]:
import matplotlib.pyplot as plt

# Histograms for numeric distributions
cols_numerical = ["alder", "matantal"]

for col in cols_numerical: 
    plt.figure(figsize=(5, 3))
    df_HE[col].dropna().hist(bins=30)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.grid(False)
    plt.show()

In [ ]:
# Barplots for categorical distributions
cols_categorical = ['team', 'sex', 'alder gruppe', 'mattype tekst', 'stain', 'undersoeger_anonymous']

for col in cols_categorical:
    plt.figure(figsize=(5, 3))
    value_counts = df_HE[col].value_counts(dropna=False)
    plt.bar(value_counts.index.astype(str), value_counts.values)
    plt.title(f"Barplot of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

sns.set_theme(style="white")

palette = {'M': '#6baed6', 'F': '#fcbba1'}

plt.figure(figsize=(10, 6))
ax = sns.histplot(
    data=df_HE,
    x='alder',
    hue='sex',
    bins=20,
    palette=palette,
    alpha=0.5,
    discrete=False,
    edgecolor='grey'
)

max_bar_height = max(p.get_height() for p in ax.patches)
ax.set_ylim(0, max_bar_height * 1.3)

# Get sexes actually present in the data
present_sexes = df_HE['sex'].dropna().unique()

# Small vertical offsets so text doesn't overlap if both exist
y_offsets = {'M': 0.95, 'F': 0.90}

for sex in present_sexes:
    ages = df_HE.loc[df_HE['sex'] == sex, 'alder'].dropna()
    if ages.empty:
        continue

    color = palette[sex]
    y_offset = y_offsets.get(sex, 1.0)

    median = np.median(ages)
    mean = ages.mean()

    # Median
    ax.axvline(median, color=color, linestyle='-', linewidth=2, alpha=0.9)
    ax.text(
        median,
        ax.get_ylim()[1] * y_offset,
        f"Median: {median:.1f}",
        color=color,
        ha='left',
        fontsize=8,
        fontweight='bold',
        backgroundcolor='white'
    )

    # Mean
    ax.axvline(mean, color=color, linestyle='--', linewidth=1.5, alpha=0.9)
    ax.text(
        mean,
        ax.get_ylim()[1] * (y_offset - 0.08),
        f"Mean: {mean:.1f}",
        color=color,
        ha='left',
        fontsize=8, 
        backgroundcolor='white'
    )

# Dynamic legend with counts
legend_handles = []
for sex in present_sexes:
    count = (df_HE['sex'] == sex).sum()
    label = 'Female' if sex == 'F' else 'Male'
    legend_handles.append(
        Patch(
            facecolor=palette[sex],
            edgecolor='grey',
            label=f'{label} (n = {count})'
        )
    )

ax.legend(handles=legend_handles, title='Sex')
ax.set_title('Age Distribution by Sex', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Number of Patients')
plt.tight_layout()
plt.show()

In [ ]:
from helper_functions import lists2tuples

df_HE = lists2tuples(df_HE)
df_HE.groupby("T_text")["M_text"].value_counts()